# Phase 2: Strategic SQL EDA
**Project:** Prescott & Vance Financial - Operations Capacity & Throughput Optimizer  
**Objective:** Execute business-focused EDA using SQL to test competing stakeholder hypotheses (resource allocation vs. structural understaffing).

In [1]:
import sqlite3
import pandas as pd
import os

# 1. Setup SQLite Database and Load Clean Fact Table
db_path = os.path.join('..', 'data', 'pv_database.db')
conn = sqlite3.connect(db_path)

csv_path = os.path.join('..', 'data', 'fact_pv_operations.csv')
df = pd.read_csv(csv_path)
df.to_sql('operations_fact', conn, if_exists='replace', index=False)

print("Database connected and fact table loaded successfully.")

Database connected and fact table loaded successfully.


### Query 1: SLA Breach Rates
**Hypothesis:** Customer complaints are rising because specific transaction types are breaching regulatory SLAs.

**Goal:** Identify which dispute types are failing at the highest rate.

In [2]:
query_sla_breach = """
SELECT 
    Dispute_Type,
    COUNT(Dispute_ID) as Total_Resolved_Cases,
    SUM(CASE WHEN SLA_Status = 'Breached' THEN 1 ELSE 0 END) as Breached_Cases,
    ROUND(CAST(SUM(CASE WHEN SLA_Status = 'Breached' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(Dispute_ID) * 100, 1) as Breach_Rate_Pct
FROM operations_fact
GROUP BY Dispute_Type
ORDER BY Breach_Rate_Pct DESC;
"""
pd.read_sql_query(query_sla_breach, conn)

,Dispute_Type,Total_Resolved_Cases,Breached_Cases,Breach_Rate_Pct
0,Merchant Dispute,174852,145736,83.3
1,Credit Card Fraud,130717,17749,13.6
2,Unauthorized Transfer,87345,11814,13.5
3,Identity Theft,43725,0,0.0


### Query 2: Hub Capacity & Efficiency
**Hypothesis:** The company is not understaffed; resources are just poorly allocated across processing hubs.

**Goal:** Evaluate hub-level throughput and SLA performance to identify structural bottlenecks.

In [3]:
query_hub_capacity = """
SELECT 
    Processing_Hub,
    COUNT(Dispute_ID) as Total_Volume,
    ROUND(AVG(Actual_Processing_Days), 1) as Avg_Handling_Time_Days,
    ROUND(CAST(SUM(CASE WHEN SLA_Status = 'Breached' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(Dispute_ID) * 100, 1) as Breach_Rate_Pct
FROM operations_fact
GROUP BY Processing_Hub
ORDER BY Total_Volume DESC;
"""
pd.read_sql_query(query_hub_capacity, conn)

,Processing_Hub,Total_Volume,Avg_Handling_Time_Days,Breach_Rate_Pct
0,Manila,218532,24.6,47.3
1,London,152731,24.6,47.1
2,New York,65376,6.5,0.0


### Query 3: The Cross-Training Insight (Merchant Disputes by Hub)
**Hypothesis:** If the New York hub processes Merchant Disputes significantly faster than Manila/London, then the bottleneck is geographic capacity, not the complexity of the dispute itself.

**Goal:** Isolate the problematic Merchant Disputes to see if the New York hub has the idle capacity and efficiency to absorb the backlog from Manila and London.

In [4]:
query_ny_idle = """
SELECT 
    Processing_Hub,
    Dispute_Type,
    ROUND(AVG(Actual_Processing_Days), 1) as Avg_Handling_Time_Days,
    ROUND(CAST(SUM(CASE WHEN SLA_Status = 'Breached' THEN 1 ELSE 0 END) AS FLOAT) / COUNT(Dispute_ID) * 100, 1) as Breach_Rate_Pct
FROM operations_fact
WHERE Dispute_Type = 'Merchant Dispute'
GROUP BY Processing_Hub, Dispute_Type
ORDER BY Breach_Rate_Pct DESC;
"""
pd.read_sql_query(query_ny_idle, conn)

,Processing_Hub,Dispute_Type,Avg_Handling_Time_Days,Breach_Rate_Pct
0,Manila,Merchant Dispute,44.2,98.2
1,London,Merchant Dispute,44.3,98.1
2,New York,Merchant Dispute,6.5,0.0


### Query 4: The 23% Surge (YoY Degradation)
**Hypothesis:** The processing time degradation is concentrated specifically in the overloaded hubs, confirming a structural allocation problem rather than a global capacity crisis.

**Goal:** Mathematically validate the COO's claim that processing times have surged year-over-year.

In [5]:
query_yoy_degradation = """
SELECT 
    Processing_Hub,
    MAX(CASE WHEN Intake_Year = '2024' THEN Avg_Days END) as Avg_2024,
    MAX(CASE WHEN Intake_Year = '2025' THEN Avg_Days END) as Avg_2025,
    ROUND((MAX(CASE WHEN Intake_Year = '2025' THEN Avg_Days END) - 
           MAX(CASE WHEN Intake_Year = '2024' THEN Avg_Days END)) / 
           MAX(CASE WHEN Intake_Year = '2024' THEN Avg_Days END) * 100, 1) as YoY_Change_Pct
FROM (
    SELECT 
        Processing_Hub,
        strftime('%Y', Intake_Timestamp) as Intake_Year,
        ROUND(AVG(Actual_Processing_Days), 1) as Avg_Days
    FROM operations_fact
    WHERE Dispute_Type = 'Merchant Dispute' 
      AND Processing_Hub IN ('Manila', 'London')
    GROUP BY Intake_Year, Processing_Hub
)
GROUP BY Processing_Hub;
"""
pd.read_sql_query(query_yoy_degradation, conn)

,Processing_Hub,Avg_2024,Avg_2025,YoY_Change_Pct
0,London,39.5,49.0,24.1
1,Manila,39.5,49.0,24.1


### Query 5: The Workflow Superiority Insight (Throughput Velocity)
**Hypothesis:** New York's 0% breach rate is not due to low volume, but due to superior operational velocity. Proving this justifies routing backlogs to NY immediately.

In [6]:
query_agent_velocity = """
SELECT
    Processing_Hub,
    COUNT(DISTINCT Processing_Agent) as Agent_Count,
    COUNT(Dispute_ID) as Total_Volume_Handled,
    ROUND(CAST(COUNT(Dispute_ID) AS FLOAT) / COUNT(DISTINCT Processing_Agent), 0) as Cases_Per_Agent,
    ROUND(AVG(Actual_Processing_Days), 1) as Avg_Handling_Time_Days
FROM operations_fact
GROUP BY Processing_Hub
ORDER BY Cases_Per_Agent DESC;
"""
pd.read_sql_query(query_agent_velocity, conn)

,Processing_Hub,Agent_Count,Total_Volume_Handled,Cases_Per_Agent,Avg_Handling_Time_Days
0,New York,3,65376,21792.0,6.5
1,Manila,15,218532,14569.0,24.6
2,London,15,152731,10182.0,24.6


In [7]:
conn.close()